In [1]:
import json
import numpy as np
import pandas as pd

with open('../dataset/results_part1.json', 'r') as f:
    part1 = json.load(f)
    
with open('../dataset/results_part2.json', 'r') as f:
    part2 = json.load(f)

In [2]:
part1_256 = next((item for item in part1 if item['dimension'] == 256))
part2_256 = next((item for item in part2 if item['dimension'] == 256))

In [3]:
recall_df = pd.DataFrame({
    'Metric' : ['Recall@1', 'Recall@5', 'Recall@10'],
    'Part 1 (Plaintext)' : [part1_256['recall_at_1'], part1_256['recall_at_5'], part1_256['recall_at_10']],
    'Part 2 (Encrypted)' : [part2_256['recall_at_1'], part2_256['recall_at_5'], part2_256['recall_at_10']]
})

print(recall_df)

      Metric  Part 1 (Plaintext)  Part 2 (Encrypted)
0   Recall@1               0.740               0.740
1   Recall@5               0.895               0.895
2  Recall@10               0.960               0.960


In [4]:
p1_queries = part1_256['queries']
p2_queries = part2_256['queries']

print(p1_queries)

{'56be4db0acb8001400a502ec': {'top_10_ids': ['56be4db0acb8001400a502ec', '56be4e1facb8001400a502f6', '56bec6573aeaaa14008c93e9', '56be5333acb8001400a5030a', '56bebbbf3aeaaa14008c9315', '56be572b3aeaaa14008c9052', '56bebad93aeaaa14008c92f9', '56bebe873aeaaa14008c933f', '56bec0dd3aeaaa14008c9357', '56be53b8acb8001400a50314'], 'scores': [0.6082297563552856, 0.55418860912323, 0.5223608613014221, 0.5067670345306396, 0.49721330404281616, 0.4913998544216156, 0.48877155780792236, 0.48670056462287903, 0.48162397742271423, 0.48015862703323364]}, '56be4e1facb8001400a502f6': {'top_10_ids': ['56be4e1facb8001400a502f6', '56beb3a03aeaaa14008c9247', '56beb4343aeaaa14008c925b', '56be59683aeaaa14008c9058', '56bec6573aeaaa14008c93e9', '56becaf93aeaaa14008c9481', '56bec8a13aeaaa14008c9435', '56bec9133aeaaa14008c9445', '56be4eafacb8001400a50302', '56beb57b3aeaaa14008c9279'], 'scores': [0.551590085029602, 0.5476073026657104, 0.5224634408950806, 0.4941879212856293, 0.45669353008270264, 0.4435688257217407, 0.

In [5]:
exact_matches = 0
mismathces = []

for q_id, q2_data in p2_queries.items():
    ids_p1 = p1_queries[q_id]['top_10_ids']
    ids_p2 = q2_data['top_10_ids']
    if ids_p1 == ids_p2:
        exact_matches += 1
    else:
        mismathces.append((q_id, ids_p1, ids_p2))
        
print(f"Exact matches: {exact_matches} out of {len(p1_queries)} queries")
print(f"Mismatches: {len(mismathces)} out of {len(p1_queries)} queries")

Exact matches: 200 out of 200 queries
Mismatches: 0 out of 200 queries


In [6]:
max_abs_diff = 0
for q_id, q2_data in p2_queries.items():
    scores_p1 = p1_queries[q_id]['scores']
    scores_p2 = q2_data['top_10_scores']
    abs_diff = np.abs(np.array(scores_p1) - np.array(scores_p2))
    max_abs_diff = max(max_abs_diff, np.max(abs_diff))
    
print(f"Maximum absolute difference in scores: {max_abs_diff}")

Maximum absolute difference in scores: 1.4648768882907781e-06


In [7]:
times_p1 = part1_256['avg_times_ms']
times_p2 = part2_256['avg_times_ms']

In [8]:
stages = [
    ("Encoding questions", times_p1['encode_q'], times_p2['encode_q']),
    ("Encrypting questions", times_p1['encrypt_q'], times_p2['encrypt_q']),
    ("Server similarity", times_p1['server_similarity'], times_p2['server_similarity']),
    ("Decrypting scores", times_p1['decrypt_scores'], times_p2['decrypt_scores']),
    ("Top-k selection", times_p1['top_k_selection'], times_p2['top_k_selection']),
    ("Fetch and decrypt documents", times_p1['fetch_and_decrypt_docs'], times_p2['fetch_and_decrypt_docs']),
]

table_rows = []
total_time_p1 = 0.0
total_time_p2 = 0.0

for name, time_p1, time_p2 in stages:
    overhead = f"{time_p2 / time_p1:.2f}" if time_p1 > 0 else "N/A"
    table_rows.append((name, f"{time_p1:.2f}", f"{time_p2:.2f}", overhead))
    total_time_p1 += time_p1
    total_time_p2 += time_p2

table_rows.append(("Total", f"{total_time_p1:.2f}", f"{total_time_p2:.2f}", f"{total_time_p2 / total_time_p1:.2f}" if total_time_p1 > 0 else "N/A"))

df = pd.DataFrame(table_rows, columns=["Stage", "Plaintext Time (ms)", "Encrypted Time (ms)", "Overhead"])
display(df)

,Stage,Plaintext Time (ms),Encrypted Time (ms),Overhead
0,Encoding questions,63.47,79.02,1.24
1,Encrypting questions,0.00,4.91,N/A
2,Server similarity,0.36,14275.07,39581.13
3,Decrypting scores,0.00,690.09,N/A
4,Top-k selection,0.11,0.17,1.55
5,Fetch and decrypt documents,0.01,0.14,25.94
6,Total,63.95,15049.40,235.34
